## 컨볼루션 커널(Convolution kernal) 예시

$$O(i, j) = (I * K)(i, j) = \sum_{m=0}^{k_h-1} \sum_{n=0}^{k_w-1} I(i+m, j+n) \cdot K(m, n)$$

여기서 각 기호의 의미는 다음과 같음.
* $O(i, j)$: 출력 행렬(Output)의 $(i, j)$ 위치의 값
* $I$: 입력 행렬 (Input Matrix, 사용자님의 5x5 행렬)
* $K$: 커널 또는 필터 (Kernel/Filter, 사용자님의 3x3 행렬)
* $k_h, k_w$: 커널의 높이와 너비 (코드에서는 3, 3)

샘플 데이터 준비

In [ ]:
import torch
import torch.nn as nn

input_matrix = torch.tensor([
    [2, 1, 0, 2, 3],
    [9, 5, 4, 2, 0],
    [2, 3, 4, 5, 6],
    [1, 2, 3, 1, 0],
    [0, 4, 4, 2, 8]
], dtype=torch.float32).unsqueeze(0).unsqueeze(0)  # (batch=1, channel=1, height=5, width=5)

print(f"입력 행렬 Shape: {input_matrix.shape}")

# 커널 (3x3 필터)
kernel = torch.tensor([
    [-1, 0, 1],
    [-1, 0, 1], 
    [-1, 0, 1]
], dtype=torch.float32).unsqueeze(0).unsqueeze(0)  # (out_channel=1, in_channel=1, height=3, width=3)

print(f"커널 Shape: {kernel.shape}")

입력 행렬 Shape: torch.Size([1, 1, 5, 5])
커널 Shape: torch.Size([1, 1, 3, 3])


컨볼루션 연산

Conv2d 파라미터 설명
- `in_channels=1`: 입력 채널 수 (흑백 이미지는 1, RGB는 3)
- `out_channels=1`: 출력 채널 수 (필터의 개수)  
- `kernel_size=3`: 커널(필터) 크기 (3x3)
- `bias=False`: 편향(bias) 사용 안함 (순수 합성곱만 계산)

In [2]:
conv_layer = nn.Conv2d(in_channels=1, out_channels=1, kernel_size=3, bias=False) # 합성곱 레이어 설정 (bias=True로 변경)
conv_layer.weight = nn.Parameter(kernel)
# bias = torch.tensor([1.0], dtype=torch.float32)  # bias 값을 1으로 설정
# conv_layer.bias = nn.Parameter(bias)

output = conv_layer(input_matrix) # 합성곱 연산 실행

print(f"출력 텐서 Shape: {output.shape}")
print("합성곱 결과:")
print(output[0, 0].detach().numpy())


출력 텐서 Shape: torch.Size([1, 1, 3, 3])
합성곱 결과:
[[-5.  0.  1.]
 [-1. -2. -5.]
 [ 8. -1.  3.]]


맥스풀링(Max pooling) 연산

$$O_{i, j} = \max_{0 \le m < k_h, \ 0 \le n < k_w} I_{i \cdot S + m, \ j \cdot S + n}$$

여기서 각 기호의 의미는 다음과 같음.
* $O_{i, j}$: 출력 행렬의 $(i, j)$ 위치 값
* $I$: 입력 행렬 (Input Feature Map)
* $k_h, k_w$: 풀링 커널의 높이와 너비
* $S$: 스트라이드 (Stride, 보폭)
* $m, n$: 커널 내부에서 움직이는 인덱스 오프셋

In [3]:
pool = nn.MaxPool2d(kernel_size=2, stride=2) # 2D max pooling 예제
pooled_output = pool(input_matrix)

print(f"풀링 결과 Shape: {pooled_output.shape}")
print("풀링 결과:")
print(pooled_output[0, 0].detach().numpy())

풀링 결과 Shape: torch.Size([1, 1, 2, 2])
풀링 결과:
[[9. 4.]
 [3. 5.]]


In [5]:
class SimpleResNet(nn.Module):
    def __init__(self):
        super(SimpleResNet, self).__init__()
        self.conv1 = nn.Conv2d(1, 1, kernel_size=3, padding=1) # padding=1 설정해 입력과 출력 shape 크기 같게 처리
        self.relu = nn.ReLU()
        self.conv2 = nn.Conv2d(1, 1, kernel_size=3, padding=1)

    def forward(self, x):
        out = self.conv1(x)
        out = self.relu(out)
        residual = out  # 레이어 출력을 잔차로 저장        
        out = self.conv2(out)
        out += residual  # 출력과 잔차 더하기
        return out

model = SimpleResNet()
output = model(input_matrix)
print("중간 레이어 잔차 연결 네트웍 출력:")
print(f"입력 Shape: {input_matrix.shape}")
print(f"출력 Shape: {output.shape}")
print(output[0, 0].detach().numpy())

중간 레이어 잔차 연결 네트웍 출력:
입력 Shape: torch.Size([1, 1, 5, 5])
출력 Shape: torch.Size([1, 1, 5, 5])
[[ 6.4005280e-01  3.0435162e+00  8.1923664e-01  5.6334019e-02
   1.3035448e+00]
 [-2.7338469e-01  4.8466182e+00  2.4086719e+00  9.9387902e-01
   3.6190746e+00]
 [-9.0280879e-01  2.5262268e+00  3.2114468e+00  2.8029151e+00
   4.3888969e+00]
 [-3.3061713e-01 -1.9974709e-03  1.9808698e+00  1.6668850e-01
   4.7440009e+00]
 [-2.0308825e-01 -6.1499578e-01  1.4095445e+00  2.1411476e+00
   2.6362488e+00]]


평탄화(Flatten) 레이어 적용

In [13]:
flatten = nn.Flatten() # Flatten 레이어 예시
flattened_output = flatten(input_matrix) # 5x5 입력을 flatten

print(f"원본 Shape: {input_matrix.shape}")      # (1, 1, 5, 5)
print(f"Flatten 후 Shape: {flattened_output.shape}")  # (1, 25)
print(f"Flatten 결과: {flattened_output.numpy()}")

원본 Shape: torch.Size([1, 1, 5, 5])
Flatten 후 Shape: torch.Size([1, 25])
Flatten 결과: [[2. 1. 0. 2. 3. 9. 5. 4. 2. 0. 2. 3. 4. 5. 6. 1. 2. 3. 1. 0. 0. 4. 4. 2.
  8.]]


In [ ]:
flatten = nn.Flatten() # Flatten 레이어 생성
out = flatten(input_matrix) 
linear = nn.Linear(in_features=out.shape[1], out_features=10) # Linear 레이어 생성
linear_output = linear(out) # 25 → 10
print(f"Linear 레이어 출력 Shape: {linear_output.shape}")
print(f"Linear 레이어 출력: {linear_output.detach().numpy()}")

## 간단한 모델 이용해 CONV 아키텍처 설계

In [12]:
# CNN + Flatten + FC 레이어 예시
class SimpleNet(nn.Module):
    def __init__(self):
        super(SimpleNet, self).__init__()
        self.conv = nn.Conv2d(1, 2, kernel_size=3)  # 5x5 → 3x3
        self.flatten = nn.Flatten()
        self.fc = nn.Linear(2 * 3 * 3, 10)  # 18 → 10

    def forward(self, x):
        x = self.conv(x)      # (1,1,5,5) → (1,2,3,3)
        x = self.flatten(x)   # (1,2,3,3) → (1,18)
        x = self.fc(x)        # (1,18) → (1,10)
        return x

# 실행
simple_net = SimpleNet()
net_output = simple_net(input_matrix)
print(f"\n전체 네트워크 출력 Shape: {net_output.shape}")
print(f"최종 출력: {net_output.detach().numpy()}")


전체 네트워크 출력 Shape: torch.Size([1, 10])
최종 출력: [[ 0.04111256 -0.22421803  2.9810069  -0.33402207 -2.243442   -0.04320517
   0.34025133 -0.5947912  -2.50786    -0.7014282 ]]


Linear 레이어 설명
- 완전 연결층(Fully Connected Layer): 모든 입력이 모든 출력에 연결
- 수식: `y = xW^T + b` (입력 x, 가중치 W, 편향 b)
- 파라미터: `weight` (입력×출력 크기), `bias` (출력 크기)
- 용도: 분류, 회귀의 최종 출력층으로 주로 사용